# LLaMA (Large Language Model Meta AI)

LLaMA is a family of large language models developed by Meta AI. They have a decoder-only structure following the original transformer model.

Some improvements over the original transformer structure are:

In [8]:
import torch 
import torch.nn as nn

## Pre-normalization

To improve the training stability, they normalize the input of each transformer sub-layer, instead of normalizing the output. They use the RMSNorm normalizing function

RMSNorm (root mean square layer normalization) definition:

$y_i = \frac{x_i}{\text{RMS}(x)} \cdot \gamma_i$, where $\text{RMS(x)} = \sqrt{\epsilon + \frac{1}{d} \sum_{j=1}^{d} x_j^2}$

Usually used in the following procedure as a pre-norm:

```python
x = RMSNorm(x)
x = x + MultiHeadAttention(x)
x = RMSNorm(x)
x = x + FeedForward(x)
```


In [9]:
# RMSNormalization
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.dim = dim
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))
    
    def forward(self, x):
        rms = torch.sqrt(self.eps + (x ** 2).mean(dim=self.dim, keepdim=True))
        return x / rms * self.scale

## SwiGLU Activation

Swish Gated Linear Unit (SwiGLU) is an activation function that combines the Swish activation function with a Gated Linear Unit (GLU). 

For a standard GLU, the input x is split into two halves:

$$
x \in \mathcal{R}^{2d} \rightarrow x_1, x_2 \in \mathcal{R}^d
$$

And the GLU then:

$$
\text{GLU}(x) = x_1 \cdot \sigma(x_2)
$$

This means one half of x gates the other half, and the output after GLU is halved.


In SwiGLU, instead of using $\sigma(x_2)$ as the gate, $SwiGLU$ uses Swish:

$$
\text{SwiGLU}(x) = x_1 \cdot \text{Swish}(x_2)
$$

where 
$$
    \text{Swish}(x) = x \cdot \sigma(x)
$$


In [10]:
class SwiGLU(nn.Module):
    def forward(self, x):
        # x: (batch_size, 2 * d). Typically from a linear layer.
        x1, x2 = x.chunk(2, dim=-1)
        return x1 * (x2 * (x2 * torch.sigmoid(x2)))

## RoPE (Rotary Position Embedding)

RoPE is a method for encoding positional information in transformer models. It improves upon traditional sinusoidal position encodings by using rotary embeddings, which allow for better generalization to longer sequences.

Suppose you have a query $q \in \mathbb{R}^d$ and a key $k \in \mathbb{R}^d$. We split the vector into 2-dimensional chunks:

$$
q = [q_1, q_2, \ldots, q_n]  \rightarrow [[q_1, q_2], [q_3, q_{4}], \ldots]
$$

For each chunk $[x, y]$, we apply a 2D rotation by an angle $\theta_p$ depending on the position $p$:

$$
\begin{bmatrix}
x' \\
y'
\end{bmatrix}
\rightarrow
\begin{bmatrix}
\cos(\theta_p) & -\sin(\theta_p) \\
\sin(\theta_p) & \cos(\theta_p)
\end{bmatrix}
\begin{bmatrix}
x \\
y
\end{bmatrix}
$$

Here $\theta_p = \frac{p}{10000^{2i/d}}$ for dimension $i \in [0, d/2]$.

This will rotate each 2D component differently depending on its dimension. When computing attention $qk^T$, the rotations allow the model to implicitly encode relative positions.

In [ ]:
def rope(q, k, start_pos=0):
    """
    q, k: tensor of shape (batch_size, seq_length, heads, head_dim)
    start_pos: absolute position offset. Positions used: [start_pos, start_pos + seq_len)
        - Training/Prefill: start_pos=0 → positions [0, 1, ..., seq_len-1]
        - Decode step t:    start_pos=t → positions [t]
    """
    batch, seq_len, n_heads, head_dim = q.shape 
    assert head_dim % 2 == 0, "head_dim must be even"

    # compute rotation angle
    dim_half = head_dim // 2
    # (dim_half,)
    inv_freq = 1.0 / (10000 ** (torch.arange(0, dim_half, dtype=torch.float32) / dim_half))
    
    # assumes q, k have the same seq_len and start_pos as in decoder-only architecture. 
    # (seq_len,) — offset by start_pos so each token gets its absolute position
    positions = torch.arange(start_pos, start_pos + seq_len, device=q.device, dtype=torch.float32)
    # (seq_len, dim_half)
    angles = positions[:, None] * inv_freq[None, :]

    # expand to batch/head dims  
    sin = angles.sin()[None, :, None, :] # (1, seq_len, 1, dim_half)
    cos = angles.cos()[None, :, None, :] # (1, seq_len, 1, dim_half)

    # split q, k into two halves
    q_1, q_2 = q.chunk(2, dim=-1)  # (batch, seq_len, heads, dim_half)
    k_1, k_2 = k.chunk(2, dim=-1)  # (batch, seq_len, heads, dim_half)

    # apply rotation
    q_1_rot = q_1 * cos - q_2 * sin
    q_2_rot = q_1 * sin + q_2 * cos
    k_1_rot = k_1 * cos - k_2 * sin
    k_2_rot = k_1 * sin + k_2 * cos

    q_rot = torch.cat([q_1_rot, q_2_rot], dim=-1)  # (batch, seq_len, heads, head_dim)
    k_rot = torch.cat([k_1_rot, k_2_rot], dim=-1)  # (batch, seq_len, heads, head_dim)

    return q_rot, k_rot


## Grouped Query Attention (GQA)

Grouped Query Attention (GQA) is a technique used to improve the efficiency of inference in transformer models, specifically by reducing the memory bandwidth required for loading keys and values (KV cache). It acts as an interpolation between Multi-Head Attention (MHA) and Multi-Query Attention (MQA).

- **Multi-Head Attention (MHA)**: Each query head has a corresponding unique key and value head. ($H$ Query heads, $H$ Key/Value heads). High quality, but high memory usage for KV cache.
- **Multi-Query Attention (MQA)**: All query heads share a single key and value head. ($H$ Query heads, 1 Key/Value head). Low memory usage, but can degrade performance.
- **Grouped Query Attention (GQA)**: Query heads are divided into $G$ groups. Each group shares a single key and value head. ($H$ Query heads, $G$ Key/Value heads, where $1 < G < H$).

**Mechanism:**
Suppose we have `n_heads` (H) for queries and we choose `n_kv_heads` (G) for keys/values.
We group the $H$ query heads into $G$ groups, resulting in $H/G$ query heads per group. All query heads in a group share the same Key and Value projection.

**Steps:**
1. **Projection**:
   - Compute $H$ Query projections.
   - Compute $G$ Key and Value projections (instead of $H$).
2. **Broadcasting (during attention)**:
   - To compute attention against $H$ query heads, the $G$ Key/Value heads are repeated (broadcasted) to match the number of query heads. Each K/V head is repeated $H/G$ times.
3. **Attention**:
   - Compute scale-dot product attention as usual using the queries and the broadcasted keys/values.

This significantly reduces the size of the KV cache compared to MHA, allowing for larger batch sizes and longer context windows during inference, while maintaining accuracy closer to MHA than MQA.

In [16]:
# some pseduo code
class GQA(nn.Module):
    def __init__(self, embed_dim, n_heads, n_kv_heads):
        super(GQA, self).__init__()
        assert embed_dim % n_heads == 0, "embed_dim must be divisible by n_heads"
        assert n_heads % n_kv_heads == 0, "n_heads must be divisible by n_kv_heads"
        
        self.embed_dim = embed_dim
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        self.n_kv_heads = n_kv_heads

        self.q_proj = nn.Linear(embed_dim, n_heads * self.head_dim)
        self.k_proj = nn.Linear(embed_dim, n_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(embed_dim, n_kv_heads * self.head_dim)
        self.o_proj = nn.Linear(embed_dim, embed_dim)

        self.attn_dropout = nn.Dropout(0.1)

    def forward(self, x, mask=None):
        batch, seq_len, _ = x.shape

        # input projection
        q_proj = self.q_proj(x)  # (batch, seq_len, n_heads * head_dim)
        k_proj = self.k_proj(x)  # (batch, seq_len, n_kv_heads * head_dim)
        v_proj = self.v_proj(x)  # (batch, seq_len, n_kv_heads * head_dim)

        # reshape
        q_proj = q_proj.view(batch, seq_len, self.n_heads, self.head_dim)  # (batch, seq_len, n_heads, head_dim)
        k_proj = k_proj.view(batch, seq_len, self.n_kv_heads, self.head_dim)  # (batch, seq_len, n_kv_heads, head_dim)
        v_proj = v_proj.view(batch, seq_len, self.n_kv_heads, self.head_dim)  # (batch, seq_len, n_kv_heads, head_dim)

        # share query within groups
        k_proj = k_proj.repeat_interleave(repeats=self.n_heads // self.n_kv_heads, dim=2)  # (batch, seq_len, n_heads, head_dim)
        v_proj = v_proj.repeat_interleave(repeats=self.n_heads // self.n_kv_heads, dim=2)  # (batch, seq_len, n_heads, head_dim)

        # attention calculation
        attn_score = torch.matmul(q_proj, k_proj.transpose(-2, -1)) / (self.head_dim ** 0.5)  # (batch, seq_len, n_heads, n_heads)
        attn_probs = torch.softmax(attn_score, dim=-1)  
        
        # apply mask if any
        if mask is not None:
            attn_probs = attn_probs.masked_fill(mask == 0, float('-inf'))
        
        # dropout 
        attn_output = self.attn_dropout(attn_output)

        # weighted sum 
        attn_output = torch.matmul(attn_probs, v_proj)  # (batch, seq_len, n_heads, head_dim)
        attn_output = attn_output.view(batch, seq_len, -1)  # (batch, seq_len, n_heads * head_dim)
        attn_output = self.o_proj(attn_output)  # (batch, seq_len, embed_dim)

        return attn_output


## KV Cache

In transformer decoding, at each step, 
- pass entire sequence through the model
- recompute Q, K ,V for all previous tokens

This waste time. KV Cache stores the K and V from past tokens, allowing the model to reuse them in future steps without recomputation.
- only compute Q, K, V from the new token
- concatenate with stored K, V from past tokens to calculate attention
- cache the new K, V for future use

Note, when using cache, only sequence length of 1 (new query at current step) is used.

In [13]:
import torch
import torch.nn as nn
from typing import Optional, Tuple

class KVCache:
    """
    Key-Value cache for efficient autoregressive generation.
    Stores K, V with shape (batch, max_seq_len, n_kv_heads, head_dim).
    """
    
    def __init__(self, max_seq_len: int, n_kv_heads: int, head_dim: int, device: torch.device):
        self.max_seq_len = max_seq_len
        self.cache_k = torch.zeros((1, max_seq_len, n_kv_heads, head_dim), device=device)
        self.cache_v = torch.zeros((1, max_seq_len, n_kv_heads, head_dim), device=device)
        self.curr_len = 0

    def update(self, key: torch.Tensor, value: torch.Tensor, start_pos: int):
        """
        Args:
           key:   (batch, seq_len, n_kv_heads, head_dim)
           value: (batch, seq_len, n_kv_heads, head_dim)
           start_pos: position offset into the cache
           
        Returns:
           full keys/values up to current length: (batch, curr_len, n_kv_heads, head_dim)
        """
        bsz, seq_len, _, _ = key.shape
        
        if self.cache_k.shape[0] < bsz:
            self.cache_k = self.cache_k.expand(bsz, -1, -1, -1).contiguous()
            self.cache_v = self.cache_v.expand(bsz, -1, -1, -1).contiguous()
            
        assert start_pos + seq_len <= self.max_seq_len

        self.cache_k[:bsz, start_pos : start_pos + seq_len] = key
        self.cache_v[:bsz, start_pos : start_pos + seq_len] = value
        self.curr_len = max(self.curr_len, start_pos + seq_len)
        
        return self.cache_k[:bsz, :self.curr_len], self.cache_v[:bsz, :self.curr_len]

    def clear(self):
        self.curr_len = 0
        self.cache_k.zero_()
        self.cache_v.zero_()
        
        
class GroupedQueryAttentionWithKVCache(nn.Module):
    """
    Grouped Query Attention with KV Cache.
    Supports MHA (n_kv_heads=n_heads), GQA (1<n_kv_heads<n_heads), MQA (n_kv_heads=1).
    Note: this mainly is used for decoder-only LLMs, thus cross attent is not considered here.
    
    Unified forward() for all modes:
        Training:  use_cache=False              → full seq, causal mask, no cache
        Prefill:   use_cache=True, start_pos=0  → full prompt, causal mask, fills cache
        Decode:    use_cache=True, start_pos=t  → single token, no mask, reads cache
    """
    def __init__(self, embed_dim, n_heads, n_kv_heads, max_seq_len=2048):
        super().__init__()
        assert embed_dim % n_heads == 0
        assert n_heads % n_kv_heads == 0
        
        self.embed_dim = embed_dim
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = embed_dim // n_heads
        self.n_rep = n_heads // n_kv_heads  # how many times to repeat each KV head
        self.max_seq_len = max_seq_len

        self.q_proj = nn.Linear(embed_dim, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        
        self.kv_cache: Optional[KVCache] = None

    def forward(self, x: torch.Tensor, start_pos: int = 0, use_cache: bool = False):
        """
        Args:
            x:         (batch, seq_len, embed_dim)
            start_pos: absolute position offset (for RoPE + cache write index)
            use_cache: enable KV cache (for inference)
        """
        bsz, seq_len, _ = x.shape

        # 1) Project → Q: (bsz, seq_len, n_heads, head_dim)
        #              K, V: (bsz, seq_len, n_kv_heads, head_dim)
        q = self.q_proj(x).view(bsz, seq_len, self.n_heads, self.head_dim)
        k = self.k_proj(x).view(bsz, seq_len, self.n_kv_heads, self.head_dim)
        v = self.v_proj(x).view(bsz, seq_len, self.n_kv_heads, self.head_dim)

        # 2) Apply RoPE BEFORE caching (cached K must have positional info baked in)
        q, k = rope(q, k, start_pos)

        # 3) KV Cache: store/retrieve with n_kv_heads (not expanded)
        if use_cache:
            if self.kv_cache is None:
                self.kv_cache = KVCache(self.max_seq_len, self.n_kv_heads, self.head_dim, x.device)
            k, v = self.kv_cache.update(k, v, start_pos)
            # k, v: (bsz, kv_len, n_kv_heads, head_dim)  where kv_len = start_pos + seq_len

        # 4) Expand KV heads to match query heads: n_kv_heads → n_heads
        #    Each KV head is repeated n_rep = n_heads // n_kv_heads times
        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=2)  # (bsz, kv_len, n_heads, head_dim)
            v = v.repeat_interleave(self.n_rep, dim=2)

        # 5) Transpose → (bsz, n_heads, seq_len/kv_len, head_dim)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # 6) Scaled dot-product attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)

        # 7) Causal mask — only when seq_len > 1
        #    Decode (seq_len=1): single query attends to ALL cached keys → no mask
        if seq_len > 1:
            kv_len = k.shape[2]
            causal_mask = torch.triu(
                torch.ones(seq_len, kv_len, dtype=torch.bool, device=x.device),
                diagonal=kv_len - seq_len + 1
            )
            scores.masked_fill_(causal_mask, float('-inf'))

        attn_weights = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn_weights, v)  # (bsz, n_heads, seq_len, head_dim)

        # 8) Reshape → output projection
        out = out.transpose(1, 2).contiguous().view(bsz, seq_len, self.embed_dim)
        return self.o_proj(out)

While the original 2017 "Attention is All You Need" paper included bias in its linear layers, most state-of-the-art models today set `bias=False`.
There are three primary reasons for this:
- Training Stability: Large-scale training is prone to "loss spikes" and numerical instability. Research (notably from Google's PaLM) found that removing bias terms in all dense kernels and layer norms significantly increased training stability at massive scales.
- Quantization Efficiency: Models are increasingly deployed using INT8 or FP8 quantization. Bias terms can introduce outliers in activations, making it harder to compress the model without losing accuracy. Removing them creates a more uniform distribution of activations, which is easier to quantize.
- Redundancy with LayerNorm: Modern Transformers use Root Mean Square Layer Normalization (RMSNorm) or standard LayerNorm before the attention layer. These normalization layers effectively re-center and re-scale the data, rendering the additive bias in the subsequent linear projection mathematically redundant.

In [14]:
# Decode with KV cache example
# Simulates: prefill a 5-token prompt, then autoregressively decode 3 new tokens

embed_dim = 64
n_heads = 8
n_kv_heads = 2  # GQA: 8 query heads, 2 KV heads
max_seq_len = 32
batch_size = 1

attn = GroupedQueryAttentionWithKVCache(embed_dim, n_heads, n_kv_heads, max_seq_len)
attn.eval()

# ─── Prefill: process entire prompt at once ───
prompt_len = 5
prompt = torch.randn(batch_size, prompt_len, embed_dim)

out_prefill = attn(prompt, start_pos=0, use_cache=True)
print(f"Prefill input:  {prompt.shape}")          # (1, 5, 64)
print(f"Prefill output: {out_prefill.shape}")      # (1, 5, 64)
print(f"Cache length after prefill: {attn.kv_cache.curr_len}")  # 5

# ─── Decode: generate tokens one at a time ───
for step in range(3):
    pos = prompt_len + step
    # In practice, this would be the embedding of the last predicted token
    new_token = torch.randn(batch_size, 1, embed_dim)
    
    out_decode = attn(new_token, start_pos=pos, use_cache=True)
    print(f"\nDecode step {step}: input {new_token.shape} at pos={pos}")
    print(f"  Output: {out_decode.shape}")          # (1, 1, 64)
    print(f"  Cache length: {attn.kv_cache.curr_len}")  # 6, 7, 8

# ─── Compare with training mode (no cache, full sequence) ───
full_seq = torch.cat([prompt, torch.randn(batch_size, 3, embed_dim)], dim=1)
attn2 = GroupedQueryAttentionWithKVCache(embed_dim, n_heads, n_kv_heads, max_seq_len)
attn2.load_state_dict(attn.state_dict())
attn2.eval()

out_train = attn2(full_seq, start_pos=0, use_cache=False)
print(f"\nTraining mode full sequence: {full_seq.shape} → {out_train.shape}")  # (1, 8, 64)

Prefill input:  torch.Size([1, 5, 64])
Prefill output: torch.Size([1, 5, 64])
Cache length after prefill: 5

Decode step 0: input torch.Size([1, 1, 64]) at pos=5
  Output: torch.Size([1, 1, 64])
  Cache length: 6

Decode step 1: input torch.Size([1, 1, 64]) at pos=6
  Output: torch.Size([1, 1, 64])
  Cache length: 7

Decode step 2: input torch.Size([1, 1, 64]) at pos=7
  Output: torch.Size([1, 1, 64])
  Cache length: 8

Training mode full sequence: torch.Size([1, 8, 64]) → torch.Size([1, 8, 64])


## LoRA (Low-Rank Adaptation)

**Low-Rank Adaptation (LoRA)** is a parameter-efficient fine-tuning (PEFT) technique for Large Language Models (LLMs). Instead of fine-tuning all the dense layers of a massive model (which is computationally expensive and requires huge storage), LoRA freezes the pre-trained model weights and injects trainable rank decomposition matrices into each layer of the Transformer architecture.

### Mathematical Formulation

Let $W_0 \in \mathbb{R}^{d \times k}$ be the fixed pre-trained weight matrix of a layer. We constrain the update $\Delta W$ to effectively have a low rank $r \ll \min(d, k)$.

The update is decomposed into two smaller matrices:
$$
\Delta W = BA
$$
where:
- $B \in \mathbb{R}^{d \times r}$
- $A \in \mathbb{R}^{r \times k}$
- $r$ is the rank hyperparameter (e.g., $r=4, 8, 16$).

**Forward Pass:**
For an input $x \in \mathbb{R}^{k}$, the modified forward pass is:
$$
h = W_0 x + \Delta W x = W_0 x + BA x
$$
This means the output is the sum of the frozen pre-trained weights' output and the trainable adapter's output.

**Initialization:**
- Matrix $A$ is initialized with random Gaussian numbers.
- Matrix $B$ is initialized with zeros.
- This ensures that at the start of training, $\Delta W = BA = 0$, so the model behaves exactly like the pre-trained model.

**Scaling:**
The update is typically scaled by a factor $\frac{\alpha}{r}$:
$$
h = W_0 x + \frac{\alpha}{r} BA x
$$
This scaling helps reduce the need to retune hyperparameters when varying $r$.

### Key Parameters

1.  **Rank ($r$)**:
    -   Controls the dimension of the low-rank matrices $A$ and $B$.
    -   A smaller $r$ means fewer trainable parameters and less memory, but potentially less expressivity.
    -   Common values: 4, 8, 16, 32. Surprisingly effective even at very low ranks ($r=1$ or $2$).

2.  **Alpha ($\alpha$)**:
    -   A scaling factor for the LoRA weights.
    -   It acts somewhat like a learning rate multiplier for the LoRA layers.
    -   Common practice is to set $\alpha$ to a multiple of $r$ (e.g., $\alpha=r$ or $\alpha=2r$).
    -   If you increase $r$, you usually adjust $\alpha$ to maintain magnitude consistency (though the $\frac{\alpha}{r}$ term handles much of this).

3.  **Target Modules**:
    -   Specifies which layers in the Transformer to apply LoRA to.
    -   Common targets: $W_q$ (Query) and $W_v$ (Value) projection matrices in the attention mechanism.
    -   Applying to all linear layers (Query, Key, Value, Output, MLP) often yields better performance but increases the parameter count slightly.

4.  **Dropout**:
    -   LoRA dropout is applied to the input of the LoRA layers (before multiplication by $A$).
    -   Standard values (e.g., 0.1) help prevent overfitting, especially with small datasets.

### Advantages
-   **Efficiency**: dramatically reduces the number of trainable parameters (often by 10,000x) and GPU memory requirements.
-   **Storage**: Checkpoints are tiny (megabytes instead of gigabytes).
-   **No Inference Latency**: After training, the matrices $BA$ can be merged back into $W_0$ ($W_{new} = W_0 + BA$), so the model architecture remains identical for inference.

## Flash Attention

The problem of standard scaled dot-product attention:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

By breaking down, we have

1. score matrix 
$$
S = \frac{QK^T}{\sqrt{d_k}}, S \in \mathbb{R}^{n \times n}
$$

2. softmax (row-wise) 
$$
P_{ij} = \frac{\exp(S_{ij})}{\sum_{j'} \exp(S_{ij'})}
$$

3. weighted sum
$$
O = P \cdot V
$$

where $Q \in \mathbb{R}^{n \times d_k}$, $K \in \mathbb{R}^{m \times d_k}$, and $V \in \mathbb{R}^{m \times d_v}$ are the query, key, and value matrices, and $d_k$ is the dimension of the keys.

### Complexity Analysis

**Time Complexity**
- $Q$, $K$, $V$ linear projection: each is a matrix multiplication $(n \times d) \times (d \times d) \rightarrow O(nd^2)$
- attention score $QK^T$ computation:
  - $(n \times d) \times (d \times n) \rightarrow O(n^2d)$
- softmax computation: $O(n^2)$
- multiply by $V$
  - $(n \times n) \times (n \times d_v) \rightarrow O(n^2d_v)$

So dominant term is O(n^2d) when n is large.

**Space Complexity**
- $Q$, $K$, $V$ storage: $O(nd_k + md_k + md_v) = O(nd)$
- attention scores: $O(n^2)$
- output: $O(n \times d_v)$

So the sum is $O(n^2 + nd)$, and the dominant term is $O(n^2)$.

### Memory Usage and Data Flow

![flash-attn](./assets/flash-attention.png)

- $Q$, $K$, $V$ are typically stored in global GPU memory.
- attention score
  - load data from global GPU memory to registers or local cache
  - compute dot product
  - store results back to global GPU memory
- softmax
  - load data from global GPU memory to registers or local cache
  - compute softmax
  - store results back to global GPU memory
- output
  - load data from global GPU memory to registers or local cache
  - compute output
  - store results back to global GPU memory

### Flash Attention

Reference: [From Online Softmax to FlashAttention](https://courses.cs.washington.edu/courses/cse599m/23sp/notes/flashattn.pdf)

FlashAttention is an optimized approach that reorganizes and fuses these steps to reduce both memory usage and global memory traffic.
It tiles the sequence dimension into blocks and computes attention in chunks in fast GPU SRAM, and never materializes $S$ in HBM.
The core idea is to derive a sequential algorithm that processes these blocks efficiently, minimizing the need for expensive global memory accesses.


**Softmax**

For a given vector $x$, softmax is defined as:

$$
\text{softmax}(x) = [\frac{\exp(x_i)}{\sum_{j} \exp(x_j)}]_{i=1}^n
$$


Usually to avoid overflow of $exp(x_j)$ when $x_j$ is large, we substract the maximum value:

$$
\text{softmax}(x) = [\frac{\exp(x_i - m)}{\sum_{j} \exp(x_j -m)}]_{i=1}^n, \text{ where } m = \max(x)
$$

The above algorithm efficiently computes the softmax values using a three-pass approach:

$$
\text{for } i = 1 \text{ to } N \\
    \hspace{100pt} m_i \leftarrow \max(m_{i-1}, x_i)
$$

$$
\text{for } i = 1 \text{ to } N \\
    \hspace{100pt} d_i \leftarrow d_{i-1} + \exp(x_i - m_N)
$$

$$
\text{for } i = 1 \text{ to } N \\
    \hspace{100pt} a_i \leftarrow \frac{\exp(x_i - m_N)}{d_N}
$$

**Online Softmax**

The problem of the original softmax is that it requires access to the entire input sequence to compute the normalization factor, which can be inefficient for long sequences or real-time applications. Online softmax addresses this issue by updating the softmax probabilities incrementally as new inputs arrive, without needing to recompute the entire softmax from scratch.

To derive an iterative computation formula, we need remove the dependency on $N$, when calculating denominator and attention if possible.

We construct the following term:

$$
\begin{aligned}
d'_i &= \sum_{j=1}^i \exp (x_j-m_i) \\
    &= (\sum_{j=1}^{i-1} \exp (x_j-m_i)) + \exp (x_i-m_i)\\
    &= (\sum_{j=1}^{i-1} \exp (x_j-m_{i-1}))\exp(m_{i-1}-m_i) + \exp (x_i-m_i) \\
    &= d'_{i-1}\exp(m_{i-1}-m_i) + \exp (x_i-m_i) \\
\end{aligned}
$$

This loop only relies on $m_{i-1}$ and $m_i$, allowing us to combine the first two passes:

$$
\text{for } i = 1 \text{ to } N \\
    \hspace{100pt} m_i \leftarrow \max(m_{i-1}, x_i) \\
    \hspace{100pt} d'_i \leftarrow d'_{i-1}\exp(m_{i-1}-m_i) + \exp (x_i-m_i)
$$

The final attention can be computed as original pass.And now we can compute softmax in two passes.

***Can we calculate the softmax() in only one pass?***

**Flash Attention**

We cannot calculate the softmax in one pass. 

But in self-attention, we focus on the weighted sum output, rather than the attention scores from softmax. Consequently, ***Can we find a way to compute the final output O in one pass?*** The answer is YES.

![self-attn-two-passes](./assets/self-attention-two-passes.png)

```latex
\begin{algorithm}[hbt!]
\caption{Multi-pass Self Attention}\label{alg:two}
$Q[k, :]: \text{the k-th row vector of } Q \text{ matrix}$;\\
$K^T[:, i]: \text{the i-th column vector of } K^T \text{ matrix}$;\\
$V[i,:]: \text{the i-th row vector of }V$;\\
$O[k,:]: \text{the k-th row of output} O$\\
$o_i: \sum_{j=1}^i a_jV[j,:], \text{ a row vector storing partial aggregation result of } A[k,:i] \times V[:i, :]$


\For {$i \leftarrow 1, N$} {

    $x_i \leftarrow Q[k,:]K^T[:, i]$ \\
    $m_i \leftarrow \max(m_{i-1}, x_i)$ \\
    $d'_i \leftarrow d'_{i-1}e^{m_{i-1} - m_i} + e^{x_i - m_i}$
}


\For {$i \leftarrow 1, N$} {
    $a_i \leftarrow \frac{e^{x_i-m_N}}{d'_N}$
    $o_i \leftarrow o_{i-1} + a_i V[i,:]$

}

$O[k, :] \leftarrow o_N$
\end{algorithm}
```



Apparently, the second pass depends on $m_N$ and $d_N$ from previous loop. We can use the same trick as in online softmax to update these values incrementally.
We introduce 

$$
\begin{aligned}
    o'_i &= \sum_{j=1}^i \frac{e^{x_j - m_i}}{d'_i}V[j,:] \\ 
         &= (\sum_{j=1}^{i-1} \frac{e^{x_j - m_i}}{d'_i}V[j,:]) + \frac{e^{x_i - m_i}}{d'_i}V[i,:] \\
         &= (\sum_{j=1}^{i-1} \frac{e^{x_j - m_{i-1}}}{d'_{i-1}}\frac{e^{x_j - m_i}}{e^{x_j - m_{i-1}}}\frac{d'_{i-1}}{d'_i}V[j,:]) + \frac{e^{x_i - m_i}}{d'_i}V[i, :]\\
         &= (\sum_{j=1}^{i-1} \frac{e^{x_j - m_{i-1}}}{d'_{i-1}}V[j,:])\frac{d'_{i-1}}{d'_i}e^{m_{i-1} - m_i} + \frac{e^{x_i - m_i}}{d'_i}V[i, :]\\
         &= o'_{i-1}\frac{d'_{i-1}}{d'_i}e^{m_{i-1} - m_i} + \frac{e^{x_i - m_i}}{d'_i}V[i, :]
\end{aligned}
$$

Now we can fuse the two passes into one.

![self-attn-single-pass](./assets/self-attention-single-pass.png)

```latex 
\begin{algorithm}[hbt!]
\caption{Single-pass Self Attention}\label{alg:two}
$Q[k, :]: \text{the k-th row vector of } Q \text{ matrix}$;\\
$K^T[:, i]: \text{the i-th column vector of } K^T \text{ matrix}$;\\
$V[i,:]: \text{the i-th row vector of }V$;\\
$O[k,:]: \text{the k-th row of output} O$\\
$o_i: \sum_{j=1}^i a_jV[j,:], \text{ a row vector storing partial aggregation result of } A[k,:i] \times V[:i, :]$


\For {$i \leftarrow 1, N$} {

    $x_i \leftarrow Q[k,:]K^T[:, i]$ \\
    $m_i \leftarrow \max(m_{i-1}, x_i)$ \\
    $d'_i \leftarrow d'_{i-1}e^{m_{i-1} - m_i} + e^{x_i - m_i}$ \\
    $o'_i \leftarrow o'_{i-1} \frac{d'_{i-1}}{d'_i} e^{m_{i-1} - m_i} + \frac{e^{x_i - m_i}}{d'_i}V[i,:]$

}

$O[k, :] \leftarrow o'_N$
\end{algorithm}

```